## Part 1: Setup

In [1]:
import time
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [2]:
# Create a SparkSession called 'spark'
spark = SparkSession.builder \
    .appName("Week 6 Project 1 Works")\
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/21 20:31:39 WARN Utils: Your hostname, Mostafizurs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.171 instead (on interface en0)
26/02/21 20:31:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 20:31:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Users
UserID::Gender::Age::Occupation::Zip-code


In [3]:
schema_users = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Occupation",IntegerType(), True),
    StructField("Zip-code", IntegerType(), True)
])

df_users = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_users) \
    .csv("./ml-1m/users.dat")

In [4]:
df_users.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: integer (nullable = true)



In [5]:
df_users.count()

6040

In [6]:
df_users.show(5)

+------+------+---+----------+--------+
|UserID|Gender|Age|Occupation|Zip-code|
+------+------+---+----------+--------+
|     1|     F|  1|        10|   48067|
|     2|     M| 56|        16|   70072|
|     3|     M| 25|        15|   55117|
|     4|     M| 45|         7|    2460|
|     5|     M| 25|        20|   55455|
+------+------+---+----------+--------+
only showing top 5 rows


## Ratings 
UserID::MovieID::Rating::Timestamp

In [7]:
schema_ratings  = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("MovieID", IntegerType(), True),
    StructField("Rating", IntegerType(), True),
    StructField("Timestamp",IntegerType(), True)
])

df_ratings = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_ratings) \
    .csv("./ml-1m/ratings.dat")

df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


In [8]:
df_ratings.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)



In [9]:
df_ratings.count()

1000209

In [10]:
df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


## Movies 
MovieID::Title::Genres

In [11]:
schema_movies  = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Genres",StringType(), True)
])

df_movies = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_movies) \
    .csv("./ml-1m/movies.dat")



In [12]:
df_movies.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



In [13]:
df_movies.count()

3883

In [14]:
df_movies.show(5)

+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


## Join the Tables

In [15]:
df_joined = df_ratings \
    .join(broadcast(df_users), on="UserID", how="inner") \
    .join(broadcast(df_movies), on="MovieID", how="inner")

In [16]:
df_joined.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



In [17]:
len(df_joined.columns)

10

In [18]:
df_joined.count()

1000209

In [19]:
df_joined.show(5)

+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|Zip-code|               Title|              Genres|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|   1193|     1|     5|978300760|     F|  1|        10|   48067|One Flew Over the...|               Drama|
|    661|     1|     3|978302109|     F|  1|        10|   48067|James and the Gia...|Animation|Childre...|
|    914|     1|     3|978301968|     F|  1|        10|   48067| My Fair Lady (1964)|     Musical|Romance|
|   3408|     1|     4|978300275|     F|  1|        10|   48067|Erin Brockovich (...|               Drama|
|   2355|     1|     5|978824291|     F|  1|        10|   48067|Bug's Life, A (1998)|Animation|Childre...|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
only showing top 5 rows


In [20]:
df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [MovieID#37, UserID#36, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4, Title#84, Genres#85]
   +- BroadcastHashJoin [MovieID#37], [MovieID#83], Inner, BuildRight, false
      :- Project [UserID#36, MovieID#37, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4]
      :  +- BroadcastHashJoin [UserID#36], [UserID#0], Inner, BuildRight, false
      :     :- Filter (isnotnull(UserID#36) AND isnotnull(MovieID#37))
      :     :  +- FileScan csv [UserID#36,MovieID#37,Rating#38,Timestamp#39] Batched: false, DataFilters: [isnotnull(UserID#36), isnotnull(MovieID#37)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Volumes/Documents/sfbu/sfbu/CS570/Week5/ml-1m/ratings.dat], PartitionFilters: [], PushedFilters: [IsNotNull(UserID), IsNotNull(MovieID)], ReadSchema: struct<UserID:int,MovieID:int,Rating:int,Timestamp:int>
      :     +- BroadcastExchange HashedRelationBroadcastMode(List(ca

## Basic Statistics

In [21]:
df_joined.describe()

DataFrame[summary: string, MovieID: string, UserID: string, Rating: string, Timestamp: string, Gender: string, Age: string, Occupation: string, Zip-code: string, Title: string, Genres: string]